# C-10 Privacy and Anonymization Protocols

**Control:** C-10 Privacy & Anonymization Protocols, MAK-AI Responsible AI Toolkit. This notebook needs a privacy-competent reviewer.

**Learning goal:** sort columns into direct identifiers, quasi-identifiers (fields that identify a person in combination, such as age, sex and village) and sensitive fields; replace identifiers with keyed pseudonyms; generalise or suppress quasi-identifiers; measure re-identification risk before and after with k-anonymity and l-diversity; and see what the data loses.

**Time to run:** about 1 minute on an ordinary laptop. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:**
- UCI Adult (Becker and Kohavi, 1996), DOI 10.24432/C5XW20, licence CC BY 4.0, 48,842 rows, from https://archive.ics.uci.edu/dataset/2/adult. It is credited here because CC BY 4.0 asks for attribution.
- Two **synthetic** columns, `name` and `phone`, generated inside the notebook from a fixed seed, because Adult has no direct identifiers. The names and numbers are invented and match no real person. The phone numbers use a toy numbering plan of eight digits.

**Run this notebook only on your own machine.** The data you anonymise never goes to the MAK-AI platform. The platform never sees it.

**A caveat before you start:** Adult is a 1994 United States census extract, and the numbers here teach a method. They say nothing about how anonymous your data is. This notebook does not judge whether what you do is adequate under your country's data protection law; it names that law only as a prompt for the people responsible.

## Linked risk areas and questions

Generated from content 5.0.0. Do not edit by hand.

Risk areas: 2

- PP-06: Privacy & Consent Violations
- IP-07: Data Integrity Compromise

Questions: 7: Q-PP-13, Q-PP-14, Q-PP-16, Q-PP-21, Q-IP-11, Q-IP-22, Q-PO-11

## When to use this notebook

Use it when you are answering questions about protecting the privacy of the people in your data and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-PP-14`: have you applied anonymization or pseudonymization techniques to protect personal data?
- `Q-PP-21`: have you established data governance policies covering access, sharing and retention?

You need a table with one row per person. If your data has free text, the optional cell near the end shows a tool for finding names and phone numbers in it.

In [ ]:
import hashlib
import hmac
import importlib.metadata
import io
import platform
import secrets
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 20261030  # the one random seed; it only makes the synthetic columns repeatable, never the key
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/static/public/2/data.csv'
DATA_SHA256 = '26b38f8623766396e2ba0d1a235a283db8ed993cc32bf32fc8aab09708c3bdf2'
DATA_FILE = 'uci-adult-2-data.csv'  # the cache file name is fixed here

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    path = CACHE_DIR / file_name
    if path.exists():
        data, source, remedy = path.read_bytes(), f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data = response.read()
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    found = hashlib.sha256(data).hexdigest()
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = path.with_name(path.name + '.part')
        partial.write_bytes(data)
        partial.replace(path)  # the cache never holds a half-written file
    return data


# To use your own file, replace the next line with:  raw = Path('your-file.csv').read_bytes()
raw = fetch_checked(DATA_URL, DATA_SHA256, DATA_FILE)
DATA_SHA256_USED = hashlib.sha256(raw).hexdigest()  # the evidence summary records the hash of the bytes used
df = pd.read_csv(io.BytesIO(raw))

# SYNTHETIC: two invented columns that stand for direct identifiers. Delete these lines for your own data.
SYNTHETIC = True
synthetic_rng = np.random.default_rng(SEED)
given = ['Amina', 'Brian', 'Cissy', 'David', 'Esther', 'Fred', 'Grace', 'Henry', 'Irene', 'Joel', 'Kevin', 'Lydia']
family = ['Akello', 'Byaruhanga', 'Chebet', 'Dhabasani', 'Ekirapa', 'Fumba', 'Gwokyalya', 'Hakim', 'Isabirye', 'Jaggwe']
df['name'] = [f'{a} {b}' for a, b in zip(synthetic_rng.choice(given, len(df)), synthetic_rng.choice(family, len(df)))]
df['phone'] = ['07' + f'{n:06d}' for n in synthetic_rng.integers(0, 10**6, len(df))]
print(f'{len(df):,} rows, {df.shape[1]} columns (two synthetic); SHA-256 of the data used: {DATA_SHA256_USED[:12]}...')

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
DIRECT_IDENTIFIERS = ['name', 'phone']     # columns that point straight at a person
PHONE_COLUMN = 'phone'                     # the direct identifier used for the hashing demonstration
QUASI_IDENTIFIERS = ['age', 'sex', 'race', 'native-country']   # fields that identify a person in combination
SENSITIVE_FIELD = 'income'                 # the field the data must not reveal about anyone
NUMERIC_TO_BAND = {'age': 10}              # numeric quasi-identifiers to turn into bands, and the width of a band
CATEGORICAL_TO_MERGE = ['race', 'native-country']   # quasi-identifiers whose rare values are merged
RARE_BELOW = 200                           # a category with fewer rows than this is merged into other
K_TARGET = 5                               # each combination of quasi-identifiers must hold at least this many people
L_TARGET = 2                               # and at least this many different sensitive values
STATISTIC_COLUMN = 'hours-per-week'        # a simple statistic whose accuracy we want to keep
STATISTIC_GROUP = 'sex'                    # compared by this group before and after

### Step 1: sort the columns

**What this does.** It labels every column with the class you gave it in the "Adapt here" cell, and shows how identifying each one is by itself:

- **Direct identifiers** point at one person: a name, a phone number, a national ID, an email address. They are removed or replaced before anything is shared.
- **Quasi-identifiers** do not identify anyone alone but do in combination: age, sex, village, job. Someone who knows these about a person can often find their row.
- **Sensitive fields** are what the data must not reveal about an individual: an illness, an income, a status.
- Everything else is **other**: check it too, because free text and rare values can identify people.

The last column of the table is the share of rows whose value in that column appears only once. It shows how quickly a column gets close to unique.

**What it shows.** Where the identifying power sits.

**What it does not show.** Whether the classification is right. That is for the people who know the data and the setting (see the "Stop and decide" cell). A column nobody thought of can identify people, such as a date of an event, or a rare job.

In [ ]:
classes = {c: 'direct identifier' for c in DIRECT_IDENTIFIERS}
classes.update({c: 'quasi-identifier' for c in QUASI_IDENTIFIERS})
classes[SENSITIVE_FIELD] = 'sensitive field'
column_table = pd.DataFrame({
    'class': [classes.get(c, 'other') for c in df.columns],
    'distinct values': [df[c].nunique() for c in df.columns],
    'share of rows whose value appears once': [float((df[c].map(df[c].value_counts()) == 1).mean()) for c in df.columns],
}, index=df.columns)
print(column_table.sort_values('class').round(3).to_string())

### Step 2: pseudonyms with a key, and why a plain hash is not enough

**What this does.** A **pseudonym** is a stand-in for an identifier that lets you link a person's rows without showing who they are. A tempting way to make one is a plain hash: the phone number goes through SHA-256 and the result looks like noise. That is not safe when the thing hashed has few possibilities. Anyone can hash every possible phone number and see which one gives your value. The first part of the cell does exactly that on a toy numbering plan of a million numbers and recovers a "hidden" number in a second or less. Real numbers have more digits, but a graphics card tries billions of guesses a second.

The fix is a **keyed hash** (HMAC-SHA256 from Python's standard `hmac` module). The key is a long random secret made fresh when the cell runs (`secrets.token_bytes(32)`). Without the key, nobody can recompute or reverse a pseudonym by guessing, and with it they can. The cell never prints the key and never writes it to a file.

**What it shows.** The difference between a hash that looks safe and one that is.

**What it does not show.** How to keep the key. A new key is made every run, so pseudonyms from different runs do not match. If you need to link files, the key must be stored somewhere safe, outside this notebook, and that is the "who holds the key" decision. Whoever has the key can re-identify everyone, so a pseudonymised file with its key is not anonymous.

In [ ]:
print('WARNING: never upload this data, its pseudonyms or the key to the MAK-AI platform. Keep them on your own machine.')

# Part 1: a plain hash of a phone number, undone by trying every number of the toy plan.
target = hashlib.sha256(df[PHONE_COLUMN].iloc[0].encode()).hexdigest()
started = time.time()
recovered = next(f'07{n:06d}' for n in range(10**6) if hashlib.sha256(f'07{n:06d}'.encode()).hexdigest() == target)
print(f'plain SHA-256 undone: tried up to 1,000,000 numbers, found one in {time.time() - started:.1f} s; same as the original: {recovered == df[PHONE_COLUMN].iloc[0]}')

# Part 2: pseudonyms with a secret key made now. Without the key the same guessing finds nothing.
PSEUDONYM_KEY = secrets.token_bytes(32)


def pseudonym(value):
    return hmac.new(PSEUDONYM_KEY, str(value).encode(), hashlib.sha256).hexdigest()[:16]


keyed_target = pseudonym(df[PHONE_COLUMN].iloc[0])
found_without_key = any(hashlib.sha256(f'07{n:06d}'.encode()).hexdigest()[:16] == keyed_target for n in range(10**6))
print(f'keyed pseudonym found by guessing without the key: {found_without_key}')

working = df.copy()
working['pseudo_id'] = [pseudonym(a) for a in working[DIRECT_IDENTIFIERS].astype(str).agg('|'.join, axis=1)]
released = working.drop(columns=DIRECT_IDENTIFIERS)
print(f'direct identifiers replaced by one pseudonym per person: {working.pseudo_id.nunique():,} different pseudonyms for {len(working):,} rows')

### Step 3: how identifiable is the data now? k-anonymity and l-diversity

**What this does.** It groups the rows by their quasi-identifiers and measures two things about the groups.

- **k-anonymity**: the size of the smallest group of rows that share the same quasi-identifier values. If k is 1, at least one person is unique, and anyone who knows their age, sex, race and country can find their row. A bigger k means each person hides in a bigger crowd.
- **l-diversity**: the smallest number of different sensitive values in any group. If l is 1, there is a group in which everyone has the same sensitive value (everyone in the group earns the same band), so knowing which group a person is in tells you the value even though you cannot tell which row is theirs.

Each is a line or two of pandas, below. No special library is needed. Teams that want more checks can install a library such as `pycanon` in their own environment; see the last cell. Pycanon is not in this notebook's locked environment because its pinned versions of pandas, numpy and scipy conflict with the rest of the notebook set.

**What it shows.** The risk of being found, from the columns you named, before anything is changed.

**What it does not show.** Risk from columns you did not name, from outside data that a determined person could match, or from a rare value you did not think of. k and l measure one kind of risk. A high k does not make data safe.

In [ ]:
def k_anonymity(frame, quasi_identifiers):
    """The size of the smallest group of rows that share the same quasi-identifier values."""
    return int(frame.groupby(quasi_identifiers, dropna=False).size().min())


def l_diversity(frame, quasi_identifiers, sensitive_field):
    """The smallest number of distinct sensitive values inside any such group."""
    return int(frame.groupby(quasi_identifiers, dropna=False)[sensitive_field].nunique().min())


def risk_table(frame):
    sizes = frame.groupby(QUASI_IDENTIFIERS, dropna=False).size()
    return {
        'rows': len(frame),
        'groups': len(sizes),
        'k (smallest group)': k_anonymity(frame, QUASI_IDENTIFIERS),
        'l (fewest sensitive values in a group)': l_diversity(frame, QUASI_IDENTIFIERS, SENSITIVE_FIELD),
        f'rows in a group smaller than {K_TARGET}': int(sizes[sizes < K_TARGET].sum()),
        'rows that are unique': int((sizes == 1).sum()),
    }


before = risk_table(released)
print(pd.Series(before).to_string())

### Step 4: generalise, merge and suppress

**What this does.** Three ways to make each person's crowd bigger, in the order that loses the least.

1. **Generalise**: turn a precise value into a band. Age 37 becomes 30 to 39. The width is in `NUMERIC_TO_BAND`.
2. **Merge rare values**: a category with fewer rows than `RARE_BELOW` becomes "other", so a rare country no longer singles someone out.
3. **Suppress**: remove the rows that are still in a group smaller than `K_TARGET`. This is the bluntest tool, and it removes real people, so the cost is counted.

Then it measures k and l again. If l is still below `L_TARGET`, it suppresses the groups in which too few different sensitive values remain, and measures once more.

**What it shows.** How much generalising and suppression it takes to reach the k you chose, on this data.

**What it does not show.** The right k. A small k is cheap and weak, and a large k is strong and costly. That judgment is the "Stop and decide" cell. And suppression is not neutral: the people whose rows are removed are the unusual ones, which may be the ones the analysis most needs.

In [ ]:
generalised = released.copy()
for column, width in NUMERIC_TO_BAND.items():
    low = (generalised[column] // width) * width
    generalised[column] = low.astype(int).astype(str) + ' to ' + (low + width - 1).astype(int).astype(str)
for column in CATEGORICAL_TO_MERGE:
    counts = generalised[column].fillna('missing').value_counts()
    rare = counts[counts < RARE_BELOW].index
    generalised[column] = generalised[column].fillna('missing').where(~generalised[column].fillna('missing').isin(rare), 'other')


def suppress_small_groups(frame, minimum_size=None, minimum_diversity=None):
    group_sizes = frame.groupby(QUASI_IDENTIFIERS, dropna=False)[SENSITIVE_FIELD].transform('size')
    diversity = frame.groupby(QUASI_IDENTIFIERS, dropna=False)[SENSITIVE_FIELD].transform('nunique')
    keep = pd.Series(True, index=frame.index)
    if minimum_size is not None:
        keep &= group_sizes >= minimum_size
    if minimum_diversity is not None:
        keep &= diversity >= minimum_diversity
    return frame[keep]


after_generalising = risk_table(generalised)
after_k = suppress_small_groups(generalised, minimum_size=K_TARGET)
after_k_risk = risk_table(after_k)
final = suppress_small_groups(after_k, minimum_diversity=L_TARGET) if after_k_risk['l (fewest sensitive values in a group)'] < L_TARGET else after_k
after = risk_table(final)

steps = pd.DataFrame({'before': before, 'generalised and merged': after_generalising,
                      f'suppressed to k >= {K_TARGET}': after_k_risk, f'then to l >= {L_TARGET}': after})
print(steps.to_string())

### Step 5: what the data lost

**What this does.** It compares a simple statistic, the average of `STATISTIC_COLUMN` for each group in `STATISTIC_GROUP`, in the original rows and in the rows that remain, and counts how many rows were removed and how many different values of each quasi-identifier are left.

**What it shows.** The cost. Anonymising is a trade between privacy and what you can learn from the data. A small change in a statistic is a good sign. A large one means the anonymised data answers a different question from the original.

**What it does not show.** The cost for analyses other than this one. Models trained on anonymised data can lose more than a simple average does, especially for the small groups that suppression removes first.

In [ ]:
original_statistic = df.groupby(STATISTIC_GROUP)[STATISTIC_COLUMN].mean()
kept_statistic = final.groupby(STATISTIC_GROUP)[STATISTIC_COLUMN].mean()
cost = pd.DataFrame({'original average': original_statistic, 'average after': kept_statistic})
cost['change'] = cost['average after'] - cost['original average']
rows_removed = len(df) - len(final)
distinct_values = pd.DataFrame({'before': [released[c].nunique() for c in QUASI_IDENTIFIERS],
                                'after': [final[c].nunique() for c in QUASI_IDENTIFIERS]}, index=QUASI_IDENTIFIERS)
print(f'rows removed: {rows_removed:,} of {len(df):,} ({rows_removed / len(df):.1%})')
print(cost.round(3).to_string())
print()
print('distinct values of each quasi-identifier:')
print(distinct_values.to_string())

### Step 6: the residual risk, in words

**What this does.** It writes the result as sentences a decision-maker can read, from the numbers above.

**What it shows.** How to say it plainly. **What it does not show.** That the sentences are the whole story: they cover only the quasi-identifiers you named.

In [ ]:
residual = (
    f"Before any change, the smallest group sharing the same {', '.join(QUASI_IDENTIFIERS)} held {before['k (smallest group)']} person(s), "
    f"and {before['rows that are unique']:,} people were unique on those columns. "
    f"After generalising, merging rare values and suppressing {rows_removed:,} rows ({rows_removed / len(df):.1%}), every combination of those "
    f"columns with at least {after['k (smallest group)']} rows in total (k = {after['k (smallest group)']}), and every such group holds at least "
    f"{after['l (fewest sensitive values in a group)']} different {SENSITIVE_FIELD} value(s) (l = {after['l (fewest sensitive values in a group)']}). "
    "This lowers the chance that someone who knows these columns finds a row. It does not remove the chance, it says nothing about other "
    "columns or outside data, and the pseudonyms are only as safe as the key."
)
print(residual)

### Further tools (optional)

**Pycanon.** `pycanon` is a free library that computes k-anonymity, l-diversity and several related measures. Teams may install it in their own environment if they want more checks than the two functions above. It is **not** part of this notebook's locked environment: its pinned versions of numpy, pandas and scipy conflict with the other notebooks' pins.

**Presidio.** Free text (notes, comments) can hold names and phone numbers in places no column list will find. Microsoft's Presidio can find some of them. The next cell is optional and is skipped in the automatic runs. Presidio needs a language model download that is large on a slow connection; install it in your own environment (`pip install presidio-analyzer presidio-anonymizer` and a spaCy model) before running it. The cell was written from the package description and has not been run by the authors of this notebook.

In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# The text below is invented.
note = 'Please call Amina Akello on 0771234567 about the follow-up visit.'
analysis = AnalyzerEngine().analyze(text=note, language='en')
print(AnonymizerEngine().anonymize(text=note, analyzer_results=analysis).text)

### Stop and decide: what k is enough here, and who holds the key

The notebook measures and removes. It cannot say how much protection is enough, and it cannot hold the key for you.

Questions to ask in the room, with someone responsible for data protection:

1. **The classification.** Did anyone who knows the data check the list of direct identifiers, quasi-identifiers and sensitive fields? Which columns nobody listed could still identify a person (a date, a rare job, a location)?
2. **What k is enough?** k = 5 is a common starting point and it is not a rule. How easy is it for someone to learn a person's quasi-identifiers? What happens to the person if they are found? A smaller crowd may be acceptable for low harm and not for high harm.
3. **Is l enough?** Even with k satisfied, a group where everyone shares a sensitive value reveals it. Is any group like that left?
4. **Who holds the key**, where is it stored, who may use it, and when is it destroyed? Whoever holds it can re-identify everyone. Is the key kept apart from the pseudonymised data?
5. **What is this data for?** If the purpose is analysis, do you need the pseudonyms at all? If you do not need to link rows, drop the identifiers and keep nothing that can be reversed.
6. **Is it allowed?** The law of your country sets duties on personal data, for example, in Uganda, the Data Protection and Privacy Act, 2019. This notebook names it only as an example and does not judge whether what you did is lawful or adequate. That is for the people responsible.

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
CLASSIFICATION_CHECKED_BY = ''   # the role of the person who checked the column classes, not a name
K_WE_ACCEPT = ''                 # the k you accept for this data, and why
L_WE_ACCEPT = ''                 # the l you accept, and why
WHO_HOLDS_THE_KEY = ''           # the role, where it is stored, who may use it, when it is destroyed
LAW_AND_REVIEW = ''              # the law and the person or body who judged adequacy

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Counts only: the class of each column, k and l at each step, and the cost. No row of the data, no pseudonym, no key.
column_table.round(4).to_csv(EVIDENCE_DIR / 'C-10_column_classification.csv')
steps.to_csv(EVIDENCE_DIR / 'C-10_k_and_l.csv')
cost.round(4).to_csv(EVIDENCE_DIR / 'C-10_cost.csv')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
classification = '\n'.join(f'- {name}: {kind}' for name, kind in classes.items())
summary = f"""# C-10 Privacy and Anonymization Protocols: evidence summary

Attach this file, with the CSV files beside it, to the decision it supports (questions Q-PP-14, Q-PP-21). It holds counts and column names only: no row of the data, no pseudonym, no key.

## Columns named
{classification}

## Method
- Direct identifiers replaced by one keyed pseudonym (HMAC-SHA256 with a random 32-byte key made when the notebook ran; the key is not recorded here or anywhere in this folder)
- Bands: {NUMERIC_TO_BAND}; rare values below {RARE_BELOW} rows merged in: {', '.join(CATEGORICAL_TO_MERGE)}
- Rows suppressed until k >= {K_TARGET}, then until l >= {L_TARGET}

## Re-identification measures
{steps.to_string()}

## Cost
- Rows removed: {rows_removed:,} of {len(df):,} ({rows_removed / len(df):.1%})
{cost.round(3).to_string()}

## Residual risk
{residual}

## Choices made in the stop-and-decide cell
- Classification checked by: {CLASSIFICATION_CHECKED_BY or 'not written yet'}
- k accepted: {K_WE_ACCEPT or 'not written yet'}
- l accepted: {L_WE_ACCEPT or 'not written yet'}
- Who holds the key: {WHO_HOLDS_THE_KEY or 'not written yet'}
- Law and review: {LAW_AND_REVIEW or 'not written yet'}

## Files beside this summary
C-10_column_classification.csv, C-10_k_and_l.csv, C-10_cost.csv

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of the data file used: {DATA_SHA256_USED}
The name and phone columns are synthetic and were made for the example.
"""
(EVIDENCE_DIR / 'C-10_summary.md').write_text(summary, encoding='utf-8')
print(summary)
print('WARNING: never upload the anonymised data, its pseudonyms or the key to the MAK-AI platform. Attach this summary only.')

## What this does not show, and where to go next

- **It does not make data anonymous.** It makes some re-identification harder, for the columns you named. Data that can be linked to outside data, or that has a column nobody listed, can still identify people.
- **k and l are one lens.** They do not cover attribute inference by a model, membership tests, or someone who already knows the person is in the data.
- **A key is a risk.** Pseudonymised data plus its key can be reversed. Treat the pair as personal data.
- **Suppression is not neutral.** It removes the unusual people, who may be the ones the analysis is about.
- **It does not judge the law.** Whether you may collect, keep, share or anonymise this data is for the people responsible, with legal advice where needed.
- **The platform must never see this data.** Keep it on your own machine. Attach only the summary this notebook writes.

Where to go next: C-03 Data Quality & Representativeness Audit (who is in the data), the datasheet template (what you recorded about origin, consent and handling), and the data-protection impact and consent templates on the control's page.